# Inflation-Aware ETF Portfolio Optimization
### Reproducible research notebook · 02

This notebook **recalculates** the portfolio models and backtests from frozen monthly ETF returns; it does not use saved strategy returns as inputs to optimization. It covers 11 baseline strategies, CPI-adjusted performance, stress periods, drift-adjusted trading costs, gold concentration constraints, and an ex-post static exposure-matched comparator.

**Research configuration:** seven Canadian-listed ETFs (five in the Core universe), 60-month trailing estimation window, 101 out-of-sample months (May 2018–September 2026), long-only fully invested weights, and historical CVaR at the 95% confidence level. The CPI-matched sample includes 99 months (June 2018–August 2026).

**How to run:** install dependencies from the repository `requirements.txt`; launch Jupyter in the repository root, open `notebooks/02_portfolio_optimization.ipynb`, and select **Restart & Run All**. If running on Google Colab, first restore/download the *full repository* to `/content/inflation-aware-etf-optimization-main` (or `/content/inflation-aware-etf-optimization`). No Google Drive connection or network price downloads are required by this notebook.

## 1. Read frozen research inputs

All paths are resolved within a local repository checkout. Inputs are the versioned CSV files in `data/processed/`. The code fails with a helpful error if only the notebook is available.

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
import cvxpy as cp
import matplotlib.pyplot as plt
from IPython.display import display

candidates = [
    Path.cwd(), Path.cwd().parent,
    Path('/content/inflation-aware-etf-optimization-main'),
    Path('/content/inflation-aware-etf-optimization'),
]
valid_roots = [p.resolve() for p in candidates
               if (p / 'data/processed/core_returns.csv').is_file()
               and (p / 'results/tables/all_11_strategy_monthly_returns.csv').is_file()]
if not valid_roots:
    raise FileNotFoundError(
        'Project CSVs not found. Open this notebook from a full checkout of '
        'inflation-aware-etf-optimization. For Colab, extract the GitHub repo ZIP '
        'into /content first; the notebook alone is not enough.'
    )
PROJECT_ROOT = valid_roots[0]
DATA_DIR = PROJECT_ROOT / 'data/processed'
TRAINING_MONTHS = 60
CVAR_CONFIDENCE = 0.95
ANNUAL_TARGETS = [0.00, 0.02, 0.04, 0.06, 0.08]

core_returns = pd.read_csv(DATA_DIR / 'core_returns.csv', index_col=0, parse_dates=True)
expanded_returns = pd.read_csv(DATA_DIR / 'expanded_returns.csv', index_col=0, parse_dates=True)
assert core_returns.shape == (161, 5)
assert expanded_returns.shape == (161, 7)
assert core_returns.index.equals(expanded_returns.index)
assert not core_returns.isna().any().any()
assert not expanded_returns.isna().any().any()
print('Repository:', PROJECT_ROOT)
print('Returns:', core_returns.shape, expanded_returns.shape)
print('CVXPY:', cp.__version__)

## 2. Optimization models

**GMV** minimizes sample portfolio variance. **Minimum CVaR** minimizes the empirical 95% expected tail loss using an auxiliary-variable linear program. **Return-constrained CVaR** adds a threshold on the **in-sample arithmetic average monthly return** converted from an annual nominal target; it is *not* a guaranteed annual realized return. All portfolios are fully invested, long-only, and without leverage.

Optimization weights are estimated using training returns only.

In [ ]:
# Stage 6 - Unit 50A
# Three core portfolio optimization models

import cvxpy as cp
import numpy as np
import pandas as pd


# Model 1: Global Minimum Variance
def optimize_gmv(returns):

    n_assets = returns.shape[1]
    cov_matrix = returns.cov().to_numpy()

    w = cp.Variable(n_assets)

    problem = cp.Problem(
        cp.Minimize(cp.quad_form(w, cov_matrix)),
        [
            cp.sum(w) == 1,
            w >= 0
        ]
    )

    # Keep the original solver selection
    problem.solve()

    if problem.status not in [
        "optimal", "optimal_inaccurate"
    ]:
        raise ValueError(problem.status)

    weights = pd.Series(
        np.asarray(w.value).ravel(),
        index=returns.columns
    )

    weights = weights.clip(lower=0)
    return weights / weights.sum()


# Model 2: Minimum CVaR
def optimize_cvar(returns, confidence=0.95):

    R = returns.to_numpy()
    T, n_assets = R.shape

    w = cp.Variable(n_assets)
    eta = cp.Variable()
    u = cp.Variable(T, nonneg=True)

    losses = -R @ w

    cvar = eta + cp.sum(u) / (
        (1 - confidence) * T
    )

    problem = cp.Problem(
        cp.Minimize(cvar),
        [
            cp.sum(w) == 1,
            w >= 0,
            u >= losses - eta
        ]
    )

    problem.solve(solver=cp.CLARABEL)

    if problem.status != "optimal":
        raise ValueError(problem.status)

    weights = pd.Series(
        np.asarray(w.value).ravel(),
        index=returns.columns
    )

    assert np.isclose(weights.sum(), 1, atol=1e-5)
    assert weights.min() >= -1e-6

    weights = weights.clip(lower=0)
    return weights / weights.sum()


# Model 3: Return-Constrained CVaR
def optimize_cvar_with_return(
    returns,
    annual_target=0.06,
    confidence=0.95
):

    R = returns.to_numpy()
    T, n_assets = R.shape

    mu = R.mean(axis=0)

    monthly_target = (
        (1 + annual_target) ** (1 / 12) - 1
    )

    w = cp.Variable(n_assets)
    eta = cp.Variable()
    u = cp.Variable(T, nonneg=True)

    losses = -R @ w

    cvar = eta + cp.sum(u) / (
        (1 - confidence) * T
    )

    problem = cp.Problem(
        cp.Minimize(cvar),
        [
            cp.sum(w) == 1,
            w >= 0,
            u >= losses - eta,
            mu @ w >= monthly_target
        ]
    )

    problem.solve(solver=cp.CLARABEL)

    if problem.status == "infeasible":
        return None

    if problem.status != "optimal":
        raise ValueError(problem.status)

    weights = pd.Series(
        np.asarray(w.value).ravel(),
        index=returns.columns
    )

    assert np.isclose(weights.sum(), 1, atol=1e-5)
    assert weights.min() >= -1e-6

    weights = weights.clip(lower=0)
    weights /= weights.sum()

    estimated_return = float(
        mu @ weights.to_numpy()
    )

    assert estimated_return >= monthly_target - 1e-6

    return weights


print("Three optimization functions defined successfully!")

### First-window model sanity check

The first training period ends in April 2018. Weight totals should equal one, and the constrained target should be feasible.

In [ ]:
# Stage 6 - Unit 50B
# Test the first 60-month training window

train = expanded_returns.iloc[:60]

print("Training period:",
      train.index.min().date(),
      "to",
      train.index.max().date())

gmv_weights = optimize_gmv(train)
cvar_weights = optimize_cvar(train)
target8_weights = optimize_cvar_with_return(
    train,
    annual_target=0.08
)

weights_table = pd.DataFrame({
    "GMV": gmv_weights,
    "Min CVaR": cvar_weights,
    "Target 8%": target8_weights
})

display(weights_table.round(4))

print("\nWeight sums:")
print(weights_table.sum().round(8))

assert target8_weights is not None
assert np.allclose(
    weights_table.sum().to_numpy(),
    1.0,
    atol=1e-6
)

print("\nFirst-window model test PASSED!")

## 3. Walk-forward out-of-sample backtesting

Each month uses only the preceding 60 monthly observations to choose weights. The next unseen month's ETF returns determine realized returns. Equal-weight portfolios are monthly rebalanced. Transaction costs are **not** deducted from the principal 11-strategy backtests.

In [ ]:
# Stage 6 - Unit 51A
# Walk-forward portfolio backtesting

import numpy as np
import pandas as pd

def walk_forward_backtest(
    returns,
    optimizer=None,
    training_months=60,
    **optimizer_kwargs
):
    """
    Monthly walk-forward backtest.

    Each test month uses only the previous
    training_months observations.
    """

    portfolio_returns = []
    portfolio_weights = []
    test_dates = []

    for i in range(training_months, len(returns)):

        # Historical training data only
        train = returns.iloc[
            i - training_months:i
        ]

        # Next, unseen test month
        test_month = returns.iloc[i]

        if optimizer is None:
            # Equal-weight benchmark
            weights = pd.Series(
                1 / returns.shape[1],
                index=returns.columns
            )
        else:
            weights = optimizer(
                train,
                **optimizer_kwargs
            )

        if weights is None:
            raise ValueError(
                f"Optimization infeasible at "
                f"{returns.index[i].date()}"
            )

        weights = weights.reindex(returns.columns)

        assert np.isclose(weights.sum(), 1, atol=1e-6)
        assert weights.min() >= -1e-6
        assert train.index.max() < test_month.name

        realized_return = float(
            np.dot(
                weights.to_numpy(),
                test_month.to_numpy()
            )
        )

        portfolio_returns.append(realized_return)
        portfolio_weights.append(
            weights.to_numpy()
        )
        test_dates.append(test_month.name)

    return (
        pd.Series(
            portfolio_returns,
            index=test_dates,
            name="Portfolio Return"
        ),
        pd.DataFrame(
            portfolio_weights,
            index=test_dates,
            columns=returns.columns
        )
    )

print("Walk-forward function defined successfully!")

In [ ]:
# Stage 6 - Unit 51B
# Recalculate all 11 portfolio strategies

strategy_returns = {}
strategy_weights = {}

strategies = {
    "Core EW": (core_returns, None, {}),
    "Expanded EW": (expanded_returns, None, {}),
    "Core GMV": (core_returns, optimize_gmv, {}),
    "Expanded GMV": (expanded_returns, optimize_gmv, {}),
    "Core CVaR": (core_returns, optimize_cvar, {}),
    "Expanded CVaR": (
        expanded_returns, optimize_cvar, {}
    ),
}

# Add five return-constrained CVaR strategies
for target in ANNUAL_TARGETS:
    name = f"Target {int(target * 100)}%"

    strategies[name] = (
        expanded_returns,
        optimize_cvar_with_return,
        {"annual_target": target}
    )

for name, (data, optimizer, kwargs) in strategies.items():

    print("Running:", name)

    result_returns, result_weights = (
        walk_forward_backtest(
            returns=data,
            optimizer=optimizer,
            training_months=TRAINING_MONTHS,
            **kwargs
        )
    )

    strategy_returns[name] = result_returns
    strategy_weights[name] = result_weights

all_recalculated_returns = pd.DataFrame(
    strategy_returns
)

print("\nBacktest completed!")
print("Shape:", all_recalculated_returns.shape)
print("First month:", all_recalculated_returns.index.min().date())
print("Last month:", all_recalculated_returns.index.max().date())
print("Missing values:",
      all_recalculated_returns.isna().sum().sum())

assert all_recalculated_returns.shape == (101, 11)
assert not all_recalculated_returns.isna().any().any()

print("\nAll 11 strategies completed successfully!")

### Numerical reproduction against frozen research outputs

The frozen CSV is used **only for comparison**, not as a model input. Floating-point solver differences are expected. Differences are checked at both the monthly-return and summary-metric levels.

In [ ]:
# Stage 6 - Unit 51C
# Compare recalculated and frozen backtest results

baseline_path = (
    PROJECT_ROOT
    / "results/tables/all_11_strategy_monthly_returns.csv"
)

baseline = pd.read_csv(
    baseline_path,
    index_col=0,
    parse_dates=True
)

print("Baseline columns:")
print(baseline.columns.tolist())

print("\nRecalculated columns:")
print(all_recalculated_returns.columns.tolist())

common_columns = (
    baseline.columns.intersection(
        all_recalculated_returns.columns
    )
)

print("\nCommon strategies:",
      len(common_columns))

if len(common_columns) > 0:

    aligned_baseline = baseline.loc[
        all_recalculated_returns.index,
        common_columns
    ]

    aligned_new = all_recalculated_returns[
        common_columns
    ]

    differences = (
        aligned_baseline - aligned_new
    ).abs()

    print("\nMaximum difference by strategy:")
    display(
        differences.max()
        .sort_values(ascending=False)
    )

    print(
        "Overall maximum difference:",
        differences.to_numpy().max()
    )

## 4. Nominal risk and return

CAGR uses compounded monthly returns; volatility uses sample standard deviation times √12; maximum drawdown compares wealth to its running peak including the initial wealth of 1; monthly historical CVaR uses a fractional 5% tail.

In [ ]:
# Stage 6 - Unit 52A
# Compare final portfolio performance metrics

import numpy as np
import pandas as pd

def calculate_performance(returns):

    returns = returns.dropna()
    T = len(returns)

    # Compound annualized return
    annual_return = (
        (1 + returns).prod() ** (12 / T) - 1
    )

    # Annualized volatility
    volatility = returns.std(ddof=1) * np.sqrt(12)

    # Maximum drawdown, including initial wealth
    wealth = (1 + returns).cumprod()
    running_peak = wealth.cummax().clip(lower=1.0)

    max_drawdown = (
        1 - wealth / running_peak
    ).max()

    # Historical monthly CVaR at 95%
    sorted_losses = np.sort(-returns.to_numpy())[::-1]

    tail_size = 0.05 * T
    full_months = int(np.floor(tail_size))
    fractional_part = tail_size - full_months

    tail_sum = sorted_losses[:full_months].sum()

    if fractional_part > 0:
        tail_sum += (
            fractional_part * sorted_losses[full_months]
        )

    cvar = tail_sum / tail_size

    return pd.Series({
        "Annual Return %": annual_return * 100,
        "Annual Volatility %": volatility * 100,
        "Maximum Drawdown %": max_drawdown * 100,
        "Monthly CVaR 95%": cvar * 100
    })


# Original frozen backtest
original_metrics = baseline.apply(
    calculate_performance
).T

# Recalculated backtest
new_metrics = all_recalculated_returns.apply(
    calculate_performance
).T

# Compare in percentage points
metric_difference = (
    original_metrics - new_metrics
).abs()

print("Recalculated performance metrics:")
display(new_metrics.round(4))

print("\nAbsolute differences (percentage points):")
display(metric_difference.round(6))

print("\nMaximum difference by metric:")
display(metric_difference.max())

print(
    "\nOverall maximum metric difference:",
    metric_difference.to_numpy().max()
)

In [ ]:
# Stage 6 - Unit 52B
# Evaluate metric-level reproducibility

# Tolerance: 0.05 percentage points
tolerance_pp = 0.05

max_difference = float(
    metric_difference.to_numpy().max()
)

print("Maximum difference:",
      round(max_difference, 8),
      "percentage points")

print("Tolerance:", tolerance_pp,
      "percentage points")

if max_difference < tolerance_pp:
    print(
        "\nPASS: Key performance metrics "
        "reproduced within tolerance!"
    )
else:
    print(
        "\nREVIEW: Some performance metrics "
        "exceed the selected tolerance."
    )
    display(
        metric_difference[
            metric_difference.gt(tolerance_pp).any(axis=1)
        ]
    )

## 5. Inflation-adjusted wealth and purchasing-power protection

Monthly real return is `(1 + nominal return) / (1 + monthly CPI inflation) − 1`. One-, three-, and five-year inflation-beating frequencies are evaluated on **overlapping** rolling windows; they are retrospective frequencies, **not** forward-looking success probabilities. CPI is used for evaluation rather than as information available to the optimizer.

In [ ]:
# Stage 6 - Unit 53A
# Inflation-adjusted portfolio returns

import pandas as pd
import numpy as np

inflation = pd.read_csv(
    PROJECT_ROOT / "data/processed/canada_monthly_inflation.csv",
    index_col=0,
    parse_dates=True
).iloc[:, 0]

inflation = inflation.astype(float)

# Match portfolio returns with available CPI months
common_dates = all_recalculated_returns.index.intersection(
    inflation.index
)

nominal_aligned = all_recalculated_returns.loc[
    common_dates
]

inflation_aligned = inflation.loc[common_dates]

# Exact real-return adjustment
real_returns = (
    (1 + nominal_aligned)
    .div(1 + inflation_aligned, axis=0)
    - 1
)

assert real_returns.shape == (99, 11)
assert not real_returns.isna().any().any()

print("CPI-matched months:", len(common_dates))
print("Start:", common_dates.min().date())
print("End:", common_dates.max().date())
print("Real return matrix:", real_returns.shape)

print("\nInflation-adjusted returns ready!")

In [ ]:
# Stage 6 - Unit 53B
# Real CAGR and historical inflation-beating frequency

def real_cagr(returns):
    return (
        (1 + returns).prod() ** (12 / len(returns))
        - 1
    )


def inflation_beating_frequency(returns, months):

    # Monthly gross real returns
    gross_returns = 1 + returns

    # Rolling compounded real returns
    rolling_real = (
        gross_returns
        .rolling(window=months, min_periods=months)
        .apply(np.prod, raw=True)
        - 1
    ).dropna()

    frequency = (rolling_real > 0).mean()

    return frequency, len(rolling_real)


records = []

for name in real_returns.columns:

    r = real_returns[name]

    freq_1y, n_1y = inflation_beating_frequency(r, 12)
    freq_3y, n_3y = inflation_beating_frequency(r, 36)
    freq_5y, n_5y = inflation_beating_frequency(r, 60)

    records.append({
        "Portfolio": name,
        "Real CAGR %": real_cagr(r) * 100,
        "Beat CPI 1Y %": freq_1y * 100,
        "Beat CPI 3Y %": freq_3y * 100,
        "Beat CPI 5Y %": freq_5y * 100,
    })

inflation_summary = (
    pd.DataFrame(records)
    .set_index("Portfolio")
)

display(inflation_summary.round(2))

print("\nRolling observation counts:")
print("1 year:", n_1y)
print("3 years:", n_3y)
print("5 years:", n_5y)

assert (n_1y, n_3y, n_5y) == (88, 64, 40)

print("\nInflation evaluation completed!")

## 6. Historical stress diagnostics

Compounded nominal portfolio returns are computed for February–March 2020 and calendar 2022. These are selected historical periods, not independent prospective tests.

In [ ]:
# Stage 6 - Unit 54
# Reproduce historical stress tests

import pandas as pd
import numpy as np

def stress_period_return(
    portfolio_returns,
    start,
    end
):
    # Select monthly returns within the period
    selected = portfolio_returns.loc[start:end]

    if selected.empty:
        raise ValueError("No observations in selected period")

    # Compound monthly returns
    cumulative_return = (
        (1 + selected).prod() - 1
    )

    return cumulative_return * 100


selected_strategies = [
    "Expanded EW",
    "Expanded CVaR",
    "Target 6%",
    "Target 8%"
]

stress_periods = {
    "COVID Feb-Mar 2020 %": (
        "2020-02-01", "2020-03-31"
    ),
    "Calendar 2022 %": (
        "2022-01-01", "2022-12-31"
    )
}

stress_results = pd.DataFrame(
    index=selected_strategies
)

for period_name, (start, end) in stress_periods.items():

    stress_results[period_name] = [
        stress_period_return(
            all_recalculated_returns[name],
            start,
            end
        )
        for name in selected_strategies
    ]

display(stress_results.round(2))

print("\nStress testing completed!")

## 7. Turnover and transaction-cost sensitivity

One-way turnover is half the absolute difference between target weights and drift-adjusted pre-rebalance holdings. Cost sensitivity uses `net monthly return = gross return − cost_rate × one_way_turnover` for 0, 5, 10 and 20 bps. Initial portfolio formation cost is omitted. This is a simplified linear friction model; it excludes spreads, taxes, and market impact.

In [ ]:
# Stage 6 - Unit 55A
# Reconstruct drift-adjusted one-way turnover

import numpy as np
import pandas as pd

def calculate_turnover(weights, asset_returns):

    weights = weights.reindex(
        columns=asset_returns.columns
    )

    asset_returns = asset_returns.reindex(
        weights.index
    )

    turnover = pd.Series(
        0.0,
        index=weights.index,
        name="Turnover"
    )

    # Start at month 2: the previous target
    # experiences the previous month's returns
    for t in range(1, len(weights)):

        previous_weights = (
            weights.iloc[t - 1].to_numpy()
        )

        previous_returns = (
            asset_returns.iloc[t - 1].to_numpy()
        )

        # Weights after the prior month's price changes
        drifted_values = (
            previous_weights * (1 + previous_returns)
        )

        drifted_weights = (
            drifted_values / drifted_values.sum()
        )

        current_weights = (
            weights.iloc[t].to_numpy()
        )

        turnover.iloc[t] = (
            0.5 * np.abs(
                current_weights - drifted_weights
            ).sum()
        )

    return turnover


turnover_strategies = [
    "Expanded EW",
    "Expanded CVaR",
    "Target 6%",
    "Target 8%"
]

turnover_results = {}
turnover_summary = {}

for name in turnover_strategies:

    turnover = calculate_turnover(
        strategy_weights[name],
        expanded_returns
    )

    turnover_results[name] = turnover

    turnover_summary[name] = {
        "Avg monthly turnover %":
            turnover.mean() * 100,
        "Annualized turnover":
            turnover.mean() * 12
    }

turnover_table = pd.DataFrame(
    turnover_summary
).T

display(turnover_table.round(3))

print("\nTurnover calculation completed!")

In [ ]:
# Stage 6 - Unit 55B
# Transaction cost sensitivity analysis

cost_levels_bps = [0, 5, 10, 20]

cost_results = []

for name in turnover_strategies:

    gross = all_recalculated_returns[name]
    turnover = turnover_results[name]

    row = {"Portfolio": name}

    for bps in cost_levels_bps:

        cost_rate = bps / 10000

        net_returns = (
            gross - cost_rate * turnover
        )

        annualized_net = (
            (1 + net_returns).prod()
            ** (12 / len(net_returns))
            - 1
        ) * 100

        row[f"{bps} bps CAGR %"] = annualized_net

    cost_results.append(row)

transaction_cost_table = (
    pd.DataFrame(cost_results)
    .set_index("Portfolio")
)

display(transaction_cost_table.round(3))

print("\nTransaction cost analysis completed!")

## 8. Gold concentration constraints

The same 8% nominal return-constrained CVaR program is rerun with `CGL-C.TO` capped at 30% or 40%. Feasibility must be checked in every training month. These are retrospective sensitivity cases rather than optimized choices of a 'best' gold cap.

In [ ]:
# Stage 6 - Unit 56A
# CVaR optimization with a gold allocation cap

def optimize_cvar_gold_cap(
    returns,
    annual_target=0.08,
    gold_cap=0.30,
    confidence=0.95
):
    R = returns.to_numpy()
    T, n_assets = R.shape

    mu = R.mean(axis=0)

    monthly_target = (
        (1 + annual_target) ** (1 / 12) - 1
    )

    gold_index = returns.columns.get_loc("CGL-C.TO")

    w = cp.Variable(n_assets)
    eta = cp.Variable()
    u = cp.Variable(T, nonneg=True)

    losses = -R @ w

    cvar = eta + cp.sum(u) / (
        (1 - confidence) * T
    )

    constraints = [
        cp.sum(w) == 1,
        w >= 0,
        u >= losses - eta,
        mu @ w >= monthly_target,
        w[gold_index] <= gold_cap
    ]

    problem = cp.Problem(
        cp.Minimize(cvar),
        constraints
    )

    problem.solve(solver=cp.CLARABEL)

    if problem.status in [
        "infeasible",
        "infeasible_inaccurate"
    ]:
        return None

    if problem.status not in [
        "optimal",
        "optimal_inaccurate"
    ]:
        raise ValueError(problem.status)

    weights = pd.Series(
        np.asarray(w.value).ravel(),
        index=returns.columns
    )

    assert abs(weights.sum() - 1) < 1e-5
    assert weights.min() >= -1e-6
    assert weights["CGL-C.TO"] <= gold_cap + 1e-6

    weights = weights.clip(lower=0)
    weights /= weights.sum()

    assert weights["CGL-C.TO"] <= gold_cap + 1e-6
    assert (
        mu @ weights.to_numpy()
        >= monthly_target - 1e-6
    )

    return weights


print("Gold-cap optimizer defined successfully!")

In [ ]:
# Stage 6 - Unit 56B
# Backtest the gold-cap strategies

gold_cap_results = {}
gold_cap_weights = {}

for cap in [0.30, 0.40]:

    name = f"Gold Cap {int(cap * 100)}%"

    print("Running:", name)

    returns_cap, weights_cap = walk_forward_backtest(
        returns=expanded_returns,
        optimizer=optimize_cvar_gold_cap,
        training_months=60,
        annual_target=0.08,
        gold_cap=cap
    )

    gold_cap_results[name] = returns_cap
    gold_cap_weights[name] = weights_cap

gold_cap_returns = pd.DataFrame(gold_cap_results)

assert gold_cap_returns.shape == (101, 2)

print("\nGold-cap backtests completed!")
print("Shape:", gold_cap_returns.shape)
print("Missing values:", gold_cap_returns.isna().sum().sum())

In [ ]:
# Stage 6 - Unit 56C
# Compare original and gold-capped Target 8%

gold_comparison_returns = pd.DataFrame({
    "Original Target 8%":
        all_recalculated_returns["Target 8%"],
    "Gold Cap 30%":
        gold_cap_returns["Gold Cap 30%"],
    "Gold Cap 40%":
        gold_cap_returns["Gold Cap 40%"]
})

gold_comparison = (
    gold_comparison_returns
    .apply(calculate_performance)
    .T
)

display(gold_comparison.round(3))

print("\nGold-cap robustness test completed!")

## 9. Ex-post exposure-matched static benchmark

The benchmark uses **full-backtest average** equity and gold exposures of Target 8%; equity exposure is split equally across four stock ETFs and the remaining exposure is assigned to XSB. Weights are fixed and **monthly rebalanced**. Since the weights are derived with hindsight, this is an explanatory ex-post benchmark, **not** an independently pre-specified investable strategy. A strong static benchmark is evidence against assuming that dynamic optimization automatically dominates simple diversification.

In [ ]:
# Stage 6 - Unit 57A
# Construct the retrospective exposure-matched benchmark

import numpy as np
import pandas as pd

equity_tickers = [
    "XIC.TO",
    "VFV.TO",
    "XEF.TO",
    "XEC.TO"
]

# Ex-post average weights from dynamic Target 8%
target8_average = strategy_weights["Target 8%"].mean()

equity_weight = float(
    target8_average[equity_tickers].sum()
)

gold_weight = float(
    target8_average["CGL-C.TO"]
)

# Match the original benchmark construction:
# equities equally split; remaining non-equity/non-gold
# allocation placed in short-term bonds.
bond_weight = 1.0 - equity_weight - gold_weight

benchmark_weights = pd.Series(
    0.0,
    index=expanded_returns.columns
)

benchmark_weights.loc[equity_tickers] = (
    equity_weight / len(equity_tickers)
)

benchmark_weights["XSB.TO"] = bond_weight
benchmark_weights["CGL-C.TO"] = gold_weight

assert np.isclose(benchmark_weights.sum(), 1.0)
assert benchmark_weights.min() >= 0

print("Static benchmark weights:")
display((benchmark_weights * 100).round(3))

print("\nAsset class allocation:")
print(f"Equities: {equity_weight * 100:.2f}%")
print(f"Short bonds: {bond_weight * 100:.2f}%")
print(f"Gold: {gold_weight * 100:.2f}%")

In [ ]:
# Stage 6 - Unit 57B
# Monthly-rebalanced static benchmark

benchmark_asset_returns = expanded_returns.loc[
    all_recalculated_returns.index,
    benchmark_weights.index
]

matched_benchmark_returns = (
    benchmark_asset_returns
    @ benchmark_weights
)

matched_benchmark_returns.name = "Matched Benchmark"

assert len(matched_benchmark_returns) == 101
assert not matched_benchmark_returns.isna().any()

# Compare nominal risk and return
nominal_comparison = pd.DataFrame({
    "Target 8%": all_recalculated_returns["Target 8%"],
    "Matched Benchmark": matched_benchmark_returns
}).apply(calculate_performance).T

display(nominal_comparison.round(3))

print("\nStatic benchmark nominal evaluation completed!")

In [ ]:
# Stage 6 - Unit 57C
# Inflation-adjusted static benchmark comparison

benchmark_nominal_aligned = matched_benchmark_returns.loc[
    common_dates
]

benchmark_real = (
    (1 + benchmark_nominal_aligned)
    / (1 + inflation_aligned)
    - 1
)

comparison_real = pd.DataFrame({
    "Target 8%": real_returns["Target 8%"],
    "Matched Benchmark": benchmark_real
})

records = []

for name in comparison_real.columns:
    r = comparison_real[name]

    records.append({
        "Portfolio": name,
        "Real CAGR %": real_cagr(r) * 100,
        "Beat CPI 1Y %":
            inflation_beating_frequency(r, 12)[0] * 100,
        "Beat CPI 3Y %":
            inflation_beating_frequency(r, 36)[0] * 100,
        "Beat CPI 5Y %":
            inflation_beating_frequency(r, 60)[0] * 100
    })

matched_real_comparison = (
    pd.DataFrame(records).set_index("Portfolio")
)

display(matched_real_comparison.round(2))

print("\nStatic benchmark real-return evaluation completed!")

## 10. Illustrative figures

Figures below are regenerated directly from the recalculated series. This notebook does not overwrite the repository's curated report figures or frozen data tables.

In [ ]:
# Figure A: nominal volatility vs CAGR for 11 OOS strategies
fig, ax = plt.subplots(figsize=(9, 5))
ax.scatter(new_metrics['Annual Volatility %'], new_metrics['Annual Return %'])
for name, row in new_metrics.iterrows():
    ax.annotate(name, (row['Annual Volatility %'], row['Annual Return %']),
                xytext=(4, 3), textcoords='offset points', fontsize=8)
ax.set(xlabel='Annualized volatility (%)', ylabel='Nominal CAGR (%)',
       title='Out-of-sample risk and return (101 months)')
ax.grid(alpha=0.25)
plt.show()

# Figure B: CPI-adjusted cumulative wealth
fig, ax = plt.subplots(figsize=(9, 5))
for name in ['Expanded EW', 'Expanded CVaR', 'Target 6%', 'Target 8%']:
    (10000 * (1 + real_returns[name]).cumprod()).plot(ax=ax, label=name)
ax.set(xlabel='Month', ylabel='Real wealth from 10,000 CAD',
       title='Inflation-adjusted growth (CPI-aligned 99 months)')
ax.legend()
ax.grid(alpha=0.25)
plt.show()

# Figure C: historical inflation-beating frequencies
plot_cols = ['Beat CPI 1Y %', 'Beat CPI 3Y %', 'Beat CPI 5Y %']
ax = inflation_summary.loc[
    ['Expanded EW', 'Expanded CVaR', 'Target 6%', 'Target 8%'], plot_cols
].plot(kind='bar', figsize=(9, 5))
ax.set(ylabel='Historical frequency (%)', title='Rolling real returns > 0')
ax.legend(title='Overlapping windows')
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 11. Final reproducibility checks and limitations

The notebook does not alter research outputs. The primary quantitative release check is that newly computed principal performance metrics differ by no more than 0.05 percentage points from the archived research outputs. For a stricter monthly return diagnostic, the maximum absolute difference is also reported (with no claim of bit-for-bit solver determinism).

**Limitations:** historical ETF selection and windows, estimation error, no taxes or investor-specific restrictions, simplified trading costs, solver tolerances, retrospective stress selection, strongly overlapping rolling CPI windows, and a hindsight-derived static benchmark. This is a research exercise, not investment advice.

In [ ]:
assert all_recalculated_returns.shape == (101, 11)
assert real_returns.shape == (99, 11)
assert gold_cap_returns.shape == (101, 2)
assert len(matched_benchmark_returns) == 101
assert (n_1y, n_3y, n_5y) == (88, 64, 40)
assert metric_difference.to_numpy().max() < 0.05
print('PASS: 11 strategies, 101 OOS months, 99 CPI-aligned months.')
print('Maximum frozen-vs-recalculated monthly absolute return difference:',
      float(differences.to_numpy().max()))
print('Maximum performance-metric difference (percentage points):',
      float(metric_difference.to_numpy().max()))
print('PASS: Nominal, real, stress, cost, concentration and matched-benchmark modules executed.')